# Goal 3 — Human perceptual correspondence  
## Stage 1B: human-reference reliability and harmonization

This notebook evaluates the human annotation reference before any signal–human correspondence model is fitted.

Primary design:

- the **69 recordings rated by all four research assistants** form the primary reliability set;
- the six acquisition-related families are evaluated separately from task-adherence labels;
- family-level and item-level presence agreement are reported together with prevalence-sensitive and prevalence-robust statistics;
- rater-specific thresholds are quantified directly;
- temporal agreement is calculated from continuous interval overlap, without imposing an arbitrary human–machine timing tolerance;
- human intervals are reconstructed by merging intervals that **overlap or touch exactly (0 ms)**;
- a 50 ms human-interval merge is retained only as a sensitivity analysis;
- no human annotation is treated as physical ground truth;
- no machine feature is loaded or modeled in this stage.


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import os
import platform
import re
import sys
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)

# ------------------------------------------------------------------
# Configurable project paths
# ------------------------------------------------------------------
def locate_analysis_root() -> Path:
    configured = os.environ.get("PAPER1_ANALYSIS_ROOT", "").strip()
    if configured:
        return Path(configured).expanduser().resolve()
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the analysis repository root")

ANALYSIS_ROOT = locate_analysis_root()

STAGE1_ROOT = (
    ANALYSIS_ROOT
    / "outputs"
    / "goal3_human_correspondence"
    / "stage1_annotation_audit"
)

OUTPUT_ROOT = (
    ANALYSIS_ROOT
    / "outputs"
    / "goal3_human_correspondence"
    / "stage1b_human_reference_reliability"
)

TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
LOG_DIR = OUTPUT_ROOT / "logs"
MANIFEST_DIR = OUTPUT_ROOT / "manifest"

for directory in [TABLE_DIR, FIGURE_DIR, LOG_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Frozen Stage-1B settings
# ------------------------------------------------------------------
EXPECTED_RATERS = ["Abbas", "Liya", "Samaana", "Samara"]
ACQUISITION_FAMILIES = ["QADD", "QGAIN", "QREV", "QCHAN", "QDIST", "QTEMP"]
ALL_SIGNAL_FAMILIES = ACQUISITION_FAMILIES + ["TASK"]

PRIMARY_HUMAN_MERGE_TOLERANCE_SEC = 0.0
SENSITIVITY_HUMAN_MERGE_TOLERANCE_SEC = 0.050

FINAL_RUN = True
N_BOOTSTRAP = 2000 if FINAL_RUN else 300
BOOTSTRAP_SEED = 20260805

ALLOW_MACHINE_COMPARISON = False

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Stage-1 root:", STAGE1_ROOT)
print("Stage-1B output:", OUTPUT_ROOT)
print("Bootstrap replicates:", N_BOOTSTRAP)
print("Machine comparison enabled:", ALLOW_MACHINE_COMPARISON)


## 1. Load the governed Stage-1 outputs

Stage 1 already verified file loading, schema integrity, JSON parsing, interval validity, and reconstruction of the 519-recording feature population. Stage 1B uses the human annotation tables only.


In [ ]:
REQUIRED_STAGE1_FILES = {
    "duplicate_audit": STAGE1_ROOT / "tables" / "humanqc_duplicate_recording_audit.csv",
    "folder_summary": STAGE1_ROOT / "tables" / "humanqc_folder_summary.csv",
    "item_summary": STAGE1_ROOT / "tables" / "humanqc_recording_item_summary.csv",
    "family_summary": STAGE1_ROOT / "tables" / "humanqc_recording_signal_family_summary.csv",
    "raw_intervals": STAGE1_ROOT / "tables" / "humanqc_raw_interval_ledger.csv",
    "stage1_manifest": STAGE1_ROOT / "manifest" / "stage1_run_manifest.json",
}

missing_files = [
    str(path)
    for path in REQUIRED_STAGE1_FILES.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Required Stage-1 outputs are missing:\n"
        + "\n".join(missing_files)
    )

duplicate_audit = pd.read_csv(REQUIRED_STAGE1_FILES["duplicate_audit"])
folder_summary = pd.read_csv(REQUIRED_STAGE1_FILES["folder_summary"])
item_summary = pd.read_csv(REQUIRED_STAGE1_FILES["item_summary"])
family_summary = pd.read_csv(REQUIRED_STAGE1_FILES["family_summary"])
raw_intervals = pd.read_csv(REQUIRED_STAGE1_FILES["raw_intervals"])

stage1_manifest = json.loads(
    REQUIRED_STAGE1_FILES["stage1_manifest"].read_text(encoding="utf-8")
)

print("Stage-1 annotation recordings:", stage1_manifest["unique_annotation_recordings"])
print("Stage-1 annotation ratings:", stage1_manifest["unique_annotation_ratings"])
display(folder_summary)


## 2. Define the complete four-rater reliability set

The primary reliability analysis requires one annotation from each of Abbas, Liya, Samaana, and Samara for the same recording. The one recording with three ratings is retained only in exported inventory and is not mixed into the primary four-rater statistics.


In [ ]:
def parse_rater_set(value: Any) -> set[str]:
    if value is None or pd.isna(value):
        return set()
    return {
        token.strip()
        for token in str(value).split("|")
        if token.strip()
    }


duplicate_audit = duplicate_audit.copy()
duplicate_audit["rater_set"] = duplicate_audit["raters"].map(parse_rater_set)

complete_four_rater = duplicate_audit.loc[
    (duplicate_audit["ratings"] == 4)
    & duplicate_audit["rater_set"].map(
        lambda values: values == set(EXPECTED_RATERS)
    )
].copy()

partial_shared = duplicate_audit.loc[
    (duplicate_audit["ratings"] >= 2)
    & ~duplicate_audit["recording_id"].isin(
        complete_four_rater["recording_id"]
    )
].copy()

if len(complete_four_rater) != 69:
    raise RuntimeError(
        "The governed complete four-rater reliability subset was expected "
        f"to contain 69 recordings, but {len(complete_four_rater)} were found."
    )

COMPLETE_RECORDING_IDS = sorted(complete_four_rater["recording_id"].unique())


def infer_reliability_subject_id(recording_id: Any) -> str:
    """
    Recover a stable subject code from reliability-subset filenames.

    Reliability files may begin with a severity-stratum prefix followed by
    '__'. The first token of the original filename is the subject code.
    """
    text = str(recording_id).strip().casefold()
    original = text.split("__")[-1]
    first_token = original.split("_")[0]
    return first_token if first_token else text


reliability_inventory = complete_four_rater[
    ["recording_id", "ratings", "raters"]
].copy()
reliability_inventory["reliability_subject_id"] = (
    reliability_inventory["recording_id"].map(
        infer_reliability_subject_id
    )
)

print("Complete four-rater recordings:", len(reliability_inventory))
print(
    "Approximate unique participants represented:",
    reliability_inventory["reliability_subject_id"].nunique(),
)
print("Other multiply rated recordings:", len(partial_shared))

display(reliability_inventory.head())
display(partial_shared)

reliability_inventory.to_csv(
    TABLE_DIR / "human_reference_complete_four_rater_inventory.csv",
    index=False,
)
partial_shared.to_csv(
    TABLE_DIR / "human_reference_partial_shared_inventory.csv",
    index=False,
)


## 3. Reliability functions

In [ ]:
def safe_divide(
    numerator: float,
    denominator: float,
) -> float:
    return (
        float(numerator / denominator)
        if denominator > 0
        else np.nan
    )


def pairwise_binary_metrics(
    first: np.ndarray,
    second: np.ndarray,
) -> dict[str, float | int]:
    first = np.asarray(first, dtype=bool)
    second = np.asarray(second, dtype=bool)

    if first.shape != second.shape:
        raise ValueError("Pairwise arrays must have identical shapes.")

    n11 = int(np.sum(first & second))
    n00 = int(np.sum(~first & ~second))
    n10 = int(np.sum(first & ~second))
    n01 = int(np.sum(~first & second))
    total = int(len(first))

    observed_agreement = safe_divide(
        n11 + n00,
        total,
    )
    positive_agreement = safe_divide(
        2 * n11,
        2 * n11 + n10 + n01,
    )
    negative_agreement = safe_divide(
        2 * n00,
        2 * n00 + n10 + n01,
    )

    prevalence_first = safe_divide(
        n11 + n10,
        total,
    )
    prevalence_second = safe_divide(
        n11 + n01,
        total,
    )

    expected_agreement = (
        prevalence_first * prevalence_second
        + (1 - prevalence_first) * (1 - prevalence_second)
    )

    cohen_kappa = (
        safe_divide(
            observed_agreement - expected_agreement,
            1 - expected_agreement,
        )
        if np.isfinite(expected_agreement)
        else np.nan
    )

    return {
        "n": total,
        "n11": n11,
        "n00": n00,
        "n10": n10,
        "n01": n01,
        "observed_agreement": observed_agreement,
        "positive_agreement": positive_agreement,
        "negative_agreement": negative_agreement,
        "cohen_kappa": cohen_kappa,
        "prevalence_first": prevalence_first,
        "prevalence_second": prevalence_second,
    }


def multirater_binary_metrics(
    matrix: np.ndarray,
) -> dict[str, float | int]:
    """
    Binary Fleiss kappa and Gwet AC1 for N units rated by R raters.

    Observed agreement is the mean proportion of agreeing rater pairs
    within each unit.
    """
    matrix = np.asarray(matrix)

    if matrix.ndim != 2:
        raise ValueError("The rating matrix must be two-dimensional.")

    if np.isnan(matrix.astype(float)).any():
        raise ValueError("The complete-case rating matrix contains missing values.")

    matrix = matrix.astype(int)

    n_units, n_raters = matrix.shape

    if n_raters < 2:
        raise ValueError("At least two raters are required.")

    category_counts = np.column_stack([
        np.sum(matrix == 0, axis=1),
        np.sum(matrix == 1, axis=1),
    ])

    agreement_per_unit = (
        np.sum(category_counts**2, axis=1) - n_raters
    ) / (
        n_raters * (n_raters - 1)
    )

    observed_agreement = float(np.mean(agreement_per_unit))
    exact_unanimous_agreement = float(
        np.mean(
            (matrix.sum(axis=1) == 0)
            | (matrix.sum(axis=1) == n_raters)
        )
    )

    marginal_probabilities = (
        category_counts.sum(axis=0)
        / (n_units * n_raters)
    )

    fleiss_expected = float(
        np.sum(marginal_probabilities**2)
    )

    fleiss_kappa = (
        safe_divide(
            observed_agreement - fleiss_expected,
            1 - fleiss_expected,
        )
    )

    # Gwet AC1 chance agreement for K=2 nominal categories:
    # sum_k p_k(1-p_k)/(K-1).
    gwet_expected = float(
        np.sum(
            marginal_probabilities
            * (1 - marginal_probabilities)
        )
    )

    gwet_ac1 = safe_divide(
        observed_agreement - gwet_expected,
        1 - gwet_expected,
    )

    pair_metrics = []
    for first_index, second_index in itertools.combinations(
        range(n_raters),
        2,
    ):
        pair_metrics.append(
            pairwise_binary_metrics(
                matrix[:, first_index],
                matrix[:, second_index],
            )
        )

    positive_values = np.asarray(
        [
            value["positive_agreement"]
            for value in pair_metrics
        ],
        dtype=float,
    )
    negative_values = np.asarray(
        [
            value["negative_agreement"]
            for value in pair_metrics
        ],
        dtype=float,
    )

    row_positive_counts = matrix.sum(axis=1)

    return {
        "units": n_units,
        "raters": n_raters,
        "positive_ratings": int(matrix.sum()),
        "any_positive_units": int(np.sum(row_positive_counts >= 1)),
        "majority_positive_units": int(
            np.sum(row_positive_counts >= math.ceil(n_raters / 2))
        ),
        "all_positive_units": int(
            np.sum(row_positive_counts == n_raters)
        ),
        "observed_agreement": observed_agreement,
        "exact_unanimous_agreement": exact_unanimous_agreement,
        "fleiss_kappa": fleiss_kappa,
        "gwet_ac1": gwet_ac1,
        "mean_pairwise_positive_agreement": (
            float(np.nanmean(positive_values))
            if np.isfinite(positive_values).any()
            else np.nan
        ),
        "mean_pairwise_negative_agreement": (
            float(np.nanmean(negative_values))
            if np.isfinite(negative_values).any()
            else np.nan
        ),
        "overall_positive_prevalence": float(matrix.mean()),
    }


def cochran_q_test(
    matrix: np.ndarray,
) -> tuple[float, float]:
    matrix = np.asarray(matrix, dtype=float)

    if matrix.ndim != 2:
        raise ValueError("Cochran Q requires a two-dimensional matrix.")

    n_units, n_raters = matrix.shape
    column_totals = matrix.sum(axis=0)
    row_totals = matrix.sum(axis=1)
    total_positive = column_totals.sum()

    denominator = (
        n_raters * total_positive
        - np.sum(row_totals**2)
    )

    if denominator <= 0:
        return np.nan, np.nan

    statistic = (
        (n_raters - 1)
        * (
            n_raters * np.sum(column_totals**2)
            - total_positive**2
        )
        / denominator
    )

    p_value = float(
        chi2.sf(statistic, df=n_raters - 1)
    )

    return float(statistic), p_value


def bh_adjust(
    p_values: Iterable[float],
) -> np.ndarray:
    values = np.asarray(list(p_values), dtype=float)
    adjusted = np.full(values.shape, np.nan, dtype=float)

    valid = np.isfinite(values)
    valid_values = values[valid]

    if valid_values.size == 0:
        return adjusted

    order = np.argsort(valid_values)
    ranked = valid_values[order]
    m = len(ranked)

    ranked_adjusted = ranked * m / np.arange(1, m + 1)
    ranked_adjusted = np.minimum.accumulate(
        ranked_adjusted[::-1]
    )[::-1]
    ranked_adjusted = np.minimum(ranked_adjusted, 1.0)

    restored = np.empty_like(ranked_adjusted)
    restored[order] = ranked_adjusted
    adjusted[valid] = restored

    return adjusted


def cluster_bootstrap_intervals(
    matrix: np.ndarray,
    cluster_ids: np.ndarray,
    metric_names: list[str],
    n_bootstrap: int,
    seed: int,
) -> dict[str, tuple[float, float]]:
    """
    Participant-cluster bootstrap.

    A selected cluster contributes all reliability recordings belonging
    to that participant. Duplicate selections are retained as separate
    bootstrap copies.
    """
    matrix = np.asarray(matrix, dtype=int)
    cluster_ids = np.asarray(cluster_ids, dtype=object)

    if len(matrix) != len(cluster_ids):
        raise ValueError("Matrix rows and cluster IDs are misaligned.")

    unique_clusters = np.asarray(
        pd.unique(cluster_ids),
        dtype=object,
    )

    row_indices_by_cluster = {
        cluster: np.flatnonzero(cluster_ids == cluster)
        for cluster in unique_clusters
    }

    rng = np.random.default_rng(seed)

    draws = {
        metric_name: np.full(n_bootstrap, np.nan)
        for metric_name in metric_names
    }

    for bootstrap_index in range(n_bootstrap):
        sampled_clusters = rng.choice(
            unique_clusters,
            size=len(unique_clusters),
            replace=True,
        )

        sampled_indices = np.concatenate([
            row_indices_by_cluster[cluster]
            for cluster in sampled_clusters
        ])

        sampled_matrix = matrix[sampled_indices]
        metrics = multirater_binary_metrics(sampled_matrix)

        for metric_name in metric_names:
            draws[metric_name][bootstrap_index] = metrics[
                metric_name
            ]

    intervals = {}

    for metric_name, values in draws.items():
        finite_values = values[np.isfinite(values)]

        if finite_values.size:
            intervals[metric_name] = (
                float(np.quantile(finite_values, 0.025)),
                float(np.quantile(finite_values, 0.975)),
            )
        else:
            intervals[metric_name] = (
                np.nan,
                np.nan,
            )

    return intervals


## 4. Build governed four-rater presence matrices

In [ ]:
subject_lookup = (
    reliability_inventory.set_index("recording_id")[
        "reliability_subject_id"
    ].to_dict()
)


def build_presence_matrix(
    frame: pd.DataFrame,
    outcome_column: str,
    outcome_value: str,
    recording_ids: list[str],
) -> tuple[np.ndarray, list[str]]:
    subset = frame.loc[
        frame["recording_id"].isin(recording_ids)
        & frame[outcome_column].eq(outcome_value)
    ].copy()

    pivot = subset.pivot(
        index="recording_id",
        columns="rater",
        values="present",
    )

    pivot = pivot.reindex(
        index=recording_ids,
        columns=EXPECTED_RATERS,
    )

    if pivot.isna().any().any():
        missing = (
            pivot.isna()
            .stack()
            .loc[lambda values: values]
            .reset_index()
        )

        raise RuntimeError(
            f"Missing four-rater presence values for {outcome_value}. "
            f"First missing rows:\n{missing.head()}"
        )

    return (
        pivot.astype(bool).to_numpy(),
        list(pivot.index),
    )


family_matrices = {}
for family in ALL_SIGNAL_FAMILIES:
    family_matrices[family] = build_presence_matrix(
        family_summary,
        "signal_family",
        family,
        COMPLETE_RECORDING_IDS,
    )

item_metadata = (
    item_summary[
        ["source_family", "signal_family", "item"]
    ]
    .drop_duplicates()
    .sort_values(
        ["signal_family", "source_family", "item"]
    )
)

item_matrices = {}
for row in item_metadata.itertuples(index=False):
    item_matrices[row.item] = build_presence_matrix(
        item_summary,
        "item",
        row.item,
        COMPLETE_RECORDING_IDS,
    )

print("Family matrices:", len(family_matrices))
print("Item matrices:", len(item_matrices))


## 5. Overall reliability across acquisition families and items

In [ ]:
# Stack recording-family combinations as reliability units while
# preserving participant clusters for bootstrap uncertainty.
family_stack_rows = []
family_stack_clusters = []

for family in ACQUISITION_FAMILIES:
    matrix, recording_order = family_matrices[family]
    family_stack_rows.append(matrix)
    family_stack_clusters.extend([
        subject_lookup[recording_id]
        for recording_id in recording_order
    ])

family_stack = np.vstack(family_stack_rows)
family_stack_clusters = np.asarray(
    family_stack_clusters,
    dtype=object,
)

item_stack_rows = []
item_stack_clusters = []

for item_name, (matrix, recording_order) in item_matrices.items():
    item_stack_rows.append(matrix)
    item_stack_clusters.extend([
        subject_lookup[recording_id]
        for recording_id in recording_order
    ])

item_stack = np.vstack(item_stack_rows)
item_stack_clusters = np.asarray(
    item_stack_clusters,
    dtype=object,
)

overall_metric_names = [
    "observed_agreement",
    "exact_unanimous_agreement",
    "fleiss_kappa",
    "gwet_ac1",
    "mean_pairwise_positive_agreement",
    "mean_pairwise_negative_agreement",
]

overall_rows = []

for level_name, matrix, clusters in [
    (
        "Six acquisition families",
        family_stack,
        family_stack_clusters,
    ),
    (
        "Specific annotation items",
        item_stack,
        item_stack_clusters,
    ),
]:
    metrics = multirater_binary_metrics(matrix)

    intervals = cluster_bootstrap_intervals(
        matrix=matrix,
        cluster_ids=clusters,
        metric_names=overall_metric_names,
        n_bootstrap=N_BOOTSTRAP,
        seed=BOOTSTRAP_SEED,
    )

    row = {
        "analysis_level": level_name,
        **metrics,
    }

    for metric_name in overall_metric_names:
        row[f"{metric_name}_ci_low"] = intervals[
            metric_name
        ][0]
        row[f"{metric_name}_ci_high"] = intervals[
            metric_name
        ][1]

    overall_rows.append(row)

overall_reliability = pd.DataFrame(overall_rows)

display(
    overall_reliability[
        [
            "analysis_level",
            "units",
            "observed_agreement",
            "fleiss_kappa",
            "gwet_ac1",
            "mean_pairwise_positive_agreement",
            "mean_pairwise_negative_agreement",
        ]
    ]
)

overall_reliability.to_csv(
    TABLE_DIR / "human_reference_overall_reliability.csv",
    index=False,
)


## 6. Family-level presence reliability

In [ ]:
FAMILY_BOOTSTRAP_METRICS = [
    "observed_agreement",
    "fleiss_kappa",
    "gwet_ac1",
    "mean_pairwise_positive_agreement",
    "mean_pairwise_negative_agreement",
]

family_rows = []

for family in ALL_SIGNAL_FAMILIES:
    matrix, recording_order = family_matrices[family]

    metrics = multirater_binary_metrics(matrix)
    cochran_statistic, cochran_p = cochran_q_test(matrix)

    clusters = np.asarray(
        [
            subject_lookup[recording_id]
            for recording_id in recording_order
        ],
        dtype=object,
    )

    intervals = cluster_bootstrap_intervals(
        matrix=matrix,
        cluster_ids=clusters,
        metric_names=FAMILY_BOOTSTRAP_METRICS,
        n_bootstrap=N_BOOTSTRAP,
        seed=BOOTSTRAP_SEED
        + ALL_SIGNAL_FAMILIES.index(family) * 100,
    )

    row = {
        "signal_family": family,
        **metrics,
        "cochran_q_statistic": cochran_statistic,
        "cochran_q_p": cochran_p,
    }

    for rater_index, rater in enumerate(EXPECTED_RATERS):
        row[f"prevalence_{rater}"] = float(
            matrix[:, rater_index].mean()
        )

    for metric_name in FAMILY_BOOTSTRAP_METRICS:
        row[f"{metric_name}_ci_low"] = intervals[
            metric_name
        ][0]
        row[f"{metric_name}_ci_high"] = intervals[
            metric_name
        ][1]

    family_rows.append(row)

family_reliability = pd.DataFrame(family_rows)
family_reliability["cochran_q_q"] = bh_adjust(
    family_reliability["cochran_q_p"]
)

family_reliability = family_reliability.sort_values(
    "signal_family",
    key=lambda values: values.map(
        {
            family: index
            for index, family in enumerate(
                ALL_SIGNAL_FAMILIES
            )
        }
    ),
)

display(
    family_reliability[
        [
            "signal_family",
            "majority_positive_units",
            "observed_agreement",
            "fleiss_kappa",
            "gwet_ac1",
            "mean_pairwise_positive_agreement",
            "mean_pairwise_negative_agreement",
            "cochran_q_q",
        ]
        + [
            f"prevalence_{rater}"
            for rater in EXPECTED_RATERS
        ]
    ]
)

family_reliability.to_csv(
    TABLE_DIR / "human_reference_family_presence_reliability.csv",
    index=False,
)


## 7. Item-level presence reliability

In [ ]:
ITEM_BOOTSTRAP_METRICS = [
    "observed_agreement",
    "fleiss_kappa",
    "gwet_ac1",
    "mean_pairwise_positive_agreement",
    "mean_pairwise_negative_agreement",
]

item_rows = []

for item_index, metadata_row in enumerate(
    item_metadata.itertuples(index=False)
):
    matrix, recording_order = item_matrices[
        metadata_row.item
    ]

    metrics = multirater_binary_metrics(matrix)
    cochran_statistic, cochran_p = cochran_q_test(matrix)

    clusters = np.asarray(
        [
            subject_lookup[recording_id]
            for recording_id in recording_order
        ],
        dtype=object,
    )

    intervals = cluster_bootstrap_intervals(
        matrix=matrix,
        cluster_ids=clusters,
        metric_names=ITEM_BOOTSTRAP_METRICS,
        n_bootstrap=N_BOOTSTRAP,
        seed=BOOTSTRAP_SEED + 1000 + item_index * 100,
    )

    row = {
        "source_family": metadata_row.source_family,
        "signal_family": metadata_row.signal_family,
        "item": metadata_row.item,
        **metrics,
        "cochran_q_statistic": cochran_statistic,
        "cochran_q_p": cochran_p,
    }

    for rater_index, rater in enumerate(EXPECTED_RATERS):
        row[f"prevalence_{rater}"] = float(
            matrix[:, rater_index].mean()
        )

    for metric_name in ITEM_BOOTSTRAP_METRICS:
        row[f"{metric_name}_ci_low"] = intervals[
            metric_name
        ][0]
        row[f"{metric_name}_ci_high"] = intervals[
            metric_name
        ][1]

    item_rows.append(row)

item_reliability = pd.DataFrame(item_rows)
item_reliability["cochran_q_q"] = bh_adjust(
    item_reliability["cochran_q_p"]
)

item_reliability = item_reliability.sort_values(
    [
        "signal_family",
        "source_family",
        "item",
    ]
)

display(
    item_reliability[
        [
            "signal_family",
            "source_family",
            "item",
            "majority_positive_units",
            "observed_agreement",
            "fleiss_kappa",
            "gwet_ac1",
            "mean_pairwise_positive_agreement",
            "mean_pairwise_negative_agreement",
            "cochran_q_q",
        ]
    ]
)

item_reliability.to_csv(
    TABLE_DIR / "human_reference_item_presence_reliability.csv",
    index=False,
)


## 8. Pairwise rater agreement

In [ ]:
def pairwise_table(
    matrix_lookup: dict[str, tuple[np.ndarray, list[str]]],
    metadata_lookup: pd.DataFrame | None = None,
) -> pd.DataFrame:
    rows = []

    item_metadata_lookup = {}
    if metadata_lookup is not None:
        item_metadata_lookup = (
            metadata_lookup.set_index("item")
            .to_dict(orient="index")
        )

    for outcome_name, (matrix, _) in matrix_lookup.items():
        for first_index, second_index in itertools.combinations(
            range(len(EXPECTED_RATERS)),
            2,
        ):
            metrics = pairwise_binary_metrics(
                matrix[:, first_index],
                matrix[:, second_index],
            )

            row = {
                "outcome": outcome_name,
                "rater_1": EXPECTED_RATERS[first_index],
                "rater_2": EXPECTED_RATERS[second_index],
                **metrics,
            }

            if outcome_name in item_metadata_lookup:
                row.update(
                    item_metadata_lookup[outcome_name]
                )

            rows.append(row)

    return pd.DataFrame(rows)


family_pairwise = pairwise_table(family_matrices)
item_pairwise = pairwise_table(
    item_matrices,
    metadata_lookup=item_metadata,
)

family_pairwise.to_csv(
    TABLE_DIR / "human_reference_pairwise_family_presence.csv",
    index=False,
)
item_pairwise.to_csv(
    TABLE_DIR / "human_reference_pairwise_item_presence.csv",
    index=False,
)

display(family_pairwise.head(20))


## 9. Each rater compared with the majority of the other three

The other-three majority is **not treated as ground truth**. It is used only to identify broader or narrower individual labeling thresholds.


In [ ]:
def rater_vs_other_three_table(
    matrix_lookup: dict[str, tuple[np.ndarray, list[str]]],
    metadata_lookup: pd.DataFrame | None = None,
) -> pd.DataFrame:
    rows = []

    item_metadata_lookup = {}
    if metadata_lookup is not None:
        item_metadata_lookup = (
            metadata_lookup.set_index("item")
            .to_dict(orient="index")
        )

    for outcome_name, (matrix, _) in matrix_lookup.items():
        for rater_index, rater in enumerate(EXPECTED_RATERS):
            other_indices = [
                index
                for index in range(len(EXPECTED_RATERS))
                if index != rater_index
            ]

            other_majority = (
                matrix[:, other_indices].sum(axis=1) >= 2
            )

            metrics = pairwise_binary_metrics(
                matrix[:, rater_index],
                other_majority,
            )

            row = {
                "outcome": outcome_name,
                "rater": rater,
                "rater_prevalence": float(
                    matrix[:, rater_index].mean()
                ),
                "other_three_majority_prevalence": float(
                    other_majority.mean()
                ),
                **metrics,
            }

            if outcome_name in item_metadata_lookup:
                row.update(
                    item_metadata_lookup[outcome_name]
                )

            rows.append(row)

    return pd.DataFrame(rows)


family_rater_vs_consensus = rater_vs_other_three_table(
    family_matrices
)
item_rater_vs_consensus = rater_vs_other_three_table(
    item_matrices,
    metadata_lookup=item_metadata,
)

family_rater_vs_consensus.to_csv(
    TABLE_DIR
    / "human_reference_family_rater_vs_other_three_majority.csv",
    index=False,
)
item_rater_vs_consensus.to_csv(
    TABLE_DIR
    / "human_reference_item_rater_vs_other_three_majority.csv",
    index=False,
)

display(family_rater_vs_consensus)


## 10. Reconstruct human intervals with 0 ms as the primary rule

This operation is limited to reconstructing human annotations split across GUI rows. It is not a human–machine alignment tolerance.


In [ ]:
def merge_intervals(
    intervals: Iterable[tuple[float, float]],
    tolerance_sec: float,
) -> list[tuple[float, float]]:
    cleaned = sorted(
        {
            (float(onset), float(offset))
            for onset, offset in intervals
            if np.isfinite(onset)
            and np.isfinite(offset)
            and offset > onset
        }
    )

    if not cleaned:
        return []

    merged = [
        [cleaned[0][0], cleaned[0][1]]
    ]

    for onset, offset in cleaned[1:]:
        previous = merged[-1]

        if onset <= previous[1] + tolerance_sec:
            previous[1] = max(
                previous[1],
                offset,
            )
        else:
            merged.append([onset, offset])

    return [
        (float(onset), float(offset))
        for onset, offset in merged
    ]


def interval_duration(
    intervals: list[tuple[float, float]],
) -> float:
    return float(
        sum(
            offset - onset
            for onset, offset in intervals
        )
    )


def intersection_duration(
    first: list[tuple[float, float]],
    second: list[tuple[float, float]],
) -> float:
    first_index = 0
    second_index = 0
    total = 0.0

    while (
        first_index < len(first)
        and second_index < len(second)
    ):
        first_onset, first_offset = first[first_index]
        second_onset, second_offset = second[second_index]

        overlap_onset = max(
            first_onset,
            second_onset,
        )
        overlap_offset = min(
            first_offset,
            second_offset,
        )

        if overlap_offset > overlap_onset:
            total += overlap_offset - overlap_onset

        if first_offset <= second_offset:
            first_index += 1
        else:
            second_index += 1

    return float(total)


valid_interval_rows = raw_intervals.loc[
    raw_intervals["recording_id"].isin(
        COMPLETE_RECORDING_IDS
    )
    & raw_intervals["valid_interval"].astype(bool)
    & ~raw_intervals["exact_duplicate"].astype(bool)
].copy()

duration_lookup = (
    item_summary.loc[
        item_summary["recording_id"].isin(
            COMPLETE_RECORDING_IDS
        )
    ]
    .groupby(
        ["recording_id", "rater"]
    )["recording_duration_sec"]
    .max()
    .to_dict()
)


def build_interval_lookup(
    tolerance_sec: float,
    outcome_level: str,
) -> dict[tuple[str, str, str], list[tuple[float, float]]]:
    if outcome_level not in {
        "signal_family",
        "item",
    }:
        raise ValueError(outcome_level)

    lookup = {}

    grouping_columns = [
        "recording_id",
        "rater",
        outcome_level,
    ]

    for keys, group in valid_interval_rows.groupby(
        grouping_columns,
        sort=False,
    ):
        lookup[
            (
                str(keys[0]),
                str(keys[1]),
                str(keys[2]),
            )
        ] = merge_intervals(
            list(
                zip(
                    group["onset_sec"],
                    group["offset_sec"],
                )
            ),
            tolerance_sec=tolerance_sec,
        )

    return lookup


interval_lookups = {}

for tolerance_label, tolerance_value in [
    (
        "primary_0ms",
        PRIMARY_HUMAN_MERGE_TOLERANCE_SEC,
    ),
    (
        "sensitivity_50ms",
        SENSITIVITY_HUMAN_MERGE_TOLERANCE_SEC,
    ),
]:
    interval_lookups[
        (
            tolerance_label,
            "signal_family",
        )
    ] = build_interval_lookup(
        tolerance_value,
        "signal_family",
    )

    interval_lookups[
        (
            tolerance_label,
            "item",
        )
    ] = build_interval_lookup(
        tolerance_value,
        "item",
    )

print("Valid complete-subset intervals:", len(valid_interval_rows))


## 11. Continuous-time agreement between raters

For each recording and rater pair:

- **temporal Dice** measures positive-time overlap;
- **temporal Jaccard** measures intersection divided by union;
- **overall time agreement** includes both jointly positive and jointly negative time.

Dice and Jaccard are summarized only when at least one rater marked the outcome. No event-boundary tolerance is imposed.


In [ ]:
def temporal_pair_rows(
    interval_lookup: dict[
        tuple[str, str, str],
        list[tuple[float, float]],
    ],
    outcome_names: Iterable[str],
    outcome_level: str,
    tolerance_label: str,
) -> pd.DataFrame:
    rows = []

    for outcome_name in outcome_names:
        for recording_id in COMPLETE_RECORDING_IDS:
            for first_rater, second_rater in itertools.combinations(
                EXPECTED_RATERS,
                2,
            ):
                first_intervals = interval_lookup.get(
                    (
                        recording_id,
                        first_rater,
                        outcome_name,
                    ),
                    [],
                )

                second_intervals = interval_lookup.get(
                    (
                        recording_id,
                        second_rater,
                        outcome_name,
                    ),
                    [],
                )

                first_duration = interval_duration(
                    first_intervals
                )
                second_duration = interval_duration(
                    second_intervals
                )
                shared_duration = intersection_duration(
                    first_intervals,
                    second_intervals,
                )

                union_duration = (
                    first_duration
                    + second_duration
                    - shared_duration
                )

                total_positive_duration = (
                    first_duration
                    + second_duration
                )

                either_positive = union_duration > 0
                both_positive = (
                    first_duration > 0
                    and second_duration > 0
                )

                temporal_dice = (
                    2 * shared_duration
                    / total_positive_duration
                    if total_positive_duration > 0
                    else np.nan
                )

                temporal_jaccard = (
                    shared_duration / union_duration
                    if union_duration > 0
                    else np.nan
                )

                recording_duration = float(
                    duration_lookup[
                        (
                            recording_id,
                            first_rater,
                        )
                    ]
                )

                second_recording_duration = float(
                    duration_lookup[
                        (
                            recording_id,
                            second_rater,
                        )
                    ]
                )

                if not np.isclose(
                    recording_duration,
                    second_recording_duration,
                    atol=1e-6,
                    rtol=0,
                ):
                    raise RuntimeError(
                        "Raters have inconsistent recording durations for "
                        f"{recording_id}."
                    )

                disagreement_duration = (
                    first_duration
                    + second_duration
                    - 2 * shared_duration
                )

                overall_time_agreement = (
                    1
                    - disagreement_duration
                    / recording_duration
                    if recording_duration > 0
                    else np.nan
                )

                rows.append({
                    "outcome_level": outcome_level,
                    "outcome": outcome_name,
                    "tolerance_label": tolerance_label,
                    "recording_id": recording_id,
                    "rater_1": first_rater,
                    "rater_2": second_rater,
                    "recording_duration_sec": recording_duration,
                    "first_positive_duration_sec": first_duration,
                    "second_positive_duration_sec": second_duration,
                    "shared_positive_duration_sec": shared_duration,
                    "union_positive_duration_sec": union_duration,
                    "either_positive": either_positive,
                    "both_positive": both_positive,
                    "temporal_dice": temporal_dice,
                    "temporal_jaccard": temporal_jaccard,
                    "overall_time_agreement": overall_time_agreement,
                })

    return pd.DataFrame(rows)


temporal_pair_tables = []

for tolerance_label in [
    "primary_0ms",
    "sensitivity_50ms",
]:
    temporal_pair_tables.append(
        temporal_pair_rows(
            interval_lookup=interval_lookups[
                (
                    tolerance_label,
                    "signal_family",
                )
            ],
            outcome_names=ALL_SIGNAL_FAMILIES,
            outcome_level="signal_family",
            tolerance_label=tolerance_label,
        )
    )

    temporal_pair_tables.append(
        temporal_pair_rows(
            interval_lookup=interval_lookups[
                (
                    tolerance_label,
                    "item",
                )
            ],
            outcome_names=item_metadata[
                "item"
            ].tolist(),
            outcome_level="item",
            tolerance_label=tolerance_label,
        )
    )

temporal_pair_detail = pd.concat(
    temporal_pair_tables,
    ignore_index=True,
)


def summarize_temporal_pairs(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    grouping_columns = [
        "outcome_level",
        "outcome",
        "tolerance_label",
    ]

    for keys, group in frame.groupby(
        grouping_columns,
        sort=False,
    ):
        positive_group = group.loc[
            group["either_positive"]
        ]

        total_shared = float(
            positive_group[
                "shared_positive_duration_sec"
            ].sum()
        )
        total_positive_duration = float(
            positive_group[
                "first_positive_duration_sec"
            ].sum()
            + positive_group[
                "second_positive_duration_sec"
            ].sum()
        )
        total_union = float(
            positive_group[
                "union_positive_duration_sec"
            ].sum()
        )

        rows.append({
            "outcome_level": keys[0],
            "outcome": keys[1],
            "tolerance_label": keys[2],
            "recording_rater_pairs": len(group),
            "either_positive_pair_recordings": int(
                group["either_positive"].sum()
            ),
            "both_positive_pair_recordings": int(
                group["both_positive"].sum()
            ),
            "pooled_temporal_dice": (
                2 * total_shared
                / total_positive_duration
                if total_positive_duration > 0
                else np.nan
            ),
            "pooled_temporal_jaccard": (
                total_shared / total_union
                if total_union > 0
                else np.nan
            ),
            "median_positive_temporal_dice": (
                float(
                    positive_group[
                        "temporal_dice"
                    ].median()
                )
                if len(positive_group)
                else np.nan
            ),
            "median_positive_temporal_jaccard": (
                float(
                    positive_group[
                        "temporal_jaccard"
                    ].median()
                )
                if len(positive_group)
                else np.nan
            ),
            "mean_overall_time_agreement": float(
                group[
                    "overall_time_agreement"
                ].mean()
            ),
            "median_overall_time_agreement": float(
                group[
                    "overall_time_agreement"
                ].median()
            ),
        })

    return pd.DataFrame(rows)


temporal_summary = summarize_temporal_pairs(
    temporal_pair_detail
)

family_temporal = temporal_summary.loc[
    temporal_summary["outcome_level"].eq(
        "signal_family"
    )
].copy()

item_temporal = temporal_summary.loc[
    temporal_summary["outcome_level"].eq(
        "item"
    )
].copy()

temporal_pair_detail.to_csv(
    TABLE_DIR / "human_reference_temporal_pair_detail.csv",
    index=False,
)
family_temporal.to_csv(
    TABLE_DIR / "human_reference_family_temporal_agreement.csv",
    index=False,
)
item_temporal.to_csv(
    TABLE_DIR / "human_reference_item_temporal_agreement.csv",
    index=False,
)

display(
    family_temporal[
        [
            "outcome",
            "tolerance_label",
            "either_positive_pair_recordings",
            "both_positive_pair_recordings",
            "pooled_temporal_dice",
            "pooled_temporal_jaccard",
            "median_positive_temporal_dice",
            "mean_overall_time_agreement",
        ]
    ]
)


## 12. Quantify the effect of the 50 ms reconstruction sensitivity

In [ ]:
def tolerance_comparison(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    primary = frame.loc[
        frame["tolerance_label"].eq(
            "primary_0ms"
        )
    ].copy()

    sensitivity = frame.loc[
        frame["tolerance_label"].eq(
            "sensitivity_50ms"
        )
    ].copy()

    merge_columns = [
        "outcome_level",
        "outcome",
    ]

    metric_columns = [
        "either_positive_pair_recordings",
        "both_positive_pair_recordings",
        "pooled_temporal_dice",
        "pooled_temporal_jaccard",
        "median_positive_temporal_dice",
        "mean_overall_time_agreement",
    ]

    comparison = primary[
        merge_columns + metric_columns
    ].merge(
        sensitivity[
            merge_columns + metric_columns
        ],
        on=merge_columns,
        suffixes=("_0ms", "_50ms"),
        validate="one_to_one",
    )

    for metric in metric_columns:
        comparison[f"{metric}_difference_50ms_minus_0ms"] = (
            comparison[f"{metric}_50ms"]
            - comparison[f"{metric}_0ms"]
        )

    return comparison


temporal_tolerance_comparison = tolerance_comparison(
    temporal_summary
)

temporal_tolerance_comparison.to_csv(
    TABLE_DIR
    / "human_reference_temporal_merge_tolerance_sensitivity.csv",
    index=False,
)

display(
    temporal_tolerance_comparison.loc[
        temporal_tolerance_comparison[
            "outcome_level"
        ].eq("signal_family")
    ]
)


## 13. Provisional human-reference roles for later signal correspondence

These roles are deliberately conservative and apply to the **human outcome**, not to the validity of any machine indicator.

A family-level outcome is classified as:

- **confirmatory candidate** when at least 10 recordings are majority-positive, Gwet AC1 is at least 0.60, and positive agreement is at least 0.50;
- **rater-sensitive exploratory candidate** when at least 10 recordings are majority-positive, Gwet AC1 is at least 0.40, and positive agreement is at least 0.25;
- **descriptive only** otherwise.

Significant Cochran-Q heterogeneity is retained as an explicit requirement for rater adjustment and leave-one-rater-out sensitivity, not used to erase an otherwise informative label.


In [ ]:
def provisional_human_role(
    majority_positive_units: int,
    gwet_ac1: float,
    positive_agreement: float,
) -> str:
    if (
        majority_positive_units >= 10
        and np.isfinite(gwet_ac1)
        and gwet_ac1 >= 0.60
        and np.isfinite(positive_agreement)
        and positive_agreement >= 0.50
    ):
        return "confirmatory_candidate"

    if (
        majority_positive_units >= 10
        and np.isfinite(gwet_ac1)
        and gwet_ac1 >= 0.40
        and np.isfinite(positive_agreement)
        and positive_agreement >= 0.25
    ):
        return "rater_sensitive_exploratory_candidate"

    return "descriptive_only"


family_decision = family_reliability.loc[
    family_reliability["signal_family"].isin(
        ACQUISITION_FAMILIES
    )
].copy()

family_decision["provisional_human_role"] = [
    provisional_human_role(
        int(row.majority_positive_units),
        float(row.gwet_ac1),
        float(
            row.mean_pairwise_positive_agreement
        ),
    )
    for row in family_decision.itertuples(index=False)
]

family_decision[
    "material_rater_prevalence_heterogeneity"
] = (
    family_decision["cochran_q_q"] < 0.05
)

family_decision[
    "later_model_requirement"
] = np.where(
    family_decision[
        "material_rater_prevalence_heterogeneity"
    ],
    (
        "Include rater fixed effects and perform "
        "leave-one-rater-out and rater-specific sensitivity analyses."
    ),
    (
        "Include rater fixed effects; standard sensitivity analyses."
    ),
)

item_decision = item_reliability.copy()
item_decision["provisional_human_role"] = [
    provisional_human_role(
        int(row.majority_positive_units),
        float(row.gwet_ac1),
        float(
            row.mean_pairwise_positive_agreement
        ),
    )
    for row in item_decision.itertuples(index=False)
]

item_decision[
    "material_rater_prevalence_heterogeneity"
] = (
    item_decision["cochran_q_q"] < 0.05
)

family_decision.to_csv(
    TABLE_DIR / "human_reference_family_role_decision.csv",
    index=False,
)
item_decision.to_csv(
    TABLE_DIR / "human_reference_item_role_decision.csv",
    index=False,
)

display(
    family_decision[
        [
            "signal_family",
            "majority_positive_units",
            "gwet_ac1",
            "mean_pairwise_positive_agreement",
            "cochran_q_q",
            "provisional_human_role",
            "material_rater_prevalence_heterogeneity",
        ]
    ]
)

display(
    item_decision.loc[
        ~item_decision[
            "provisional_human_role"
        ].eq("descriptive_only")
    ][
        [
            "signal_family",
            "source_family",
            "item",
            "majority_positive_units",
            "gwet_ac1",
            "mean_pairwise_positive_agreement",
            "cochran_q_q",
            "provisional_human_role",
        ]
    ]
)


## 14. Audit figures

In [ ]:
mpl.rcParams.update({
    "font.family": ["Arial", "DejaVu Sans"],
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


def save_current_figure(
    stem: str,
) -> None:
    for suffix in [".png", ".svg", ".pdf"]:
        plt.savefig(
            FIGURE_DIR / f"{stem}{suffix}",
            dpi=400 if suffix == ".png" else None,
            bbox_inches="tight",
        )


# --------------------------------------------------------------
# Figure 1: family prevalence by rater
# --------------------------------------------------------------
prevalence_long = family_reliability.melt(
    id_vars=["signal_family"],
    value_vars=[
        f"prevalence_{rater}"
        for rater in EXPECTED_RATERS
    ],
    var_name="rater",
    value_name="prevalence",
)
prevalence_long["rater"] = (
    prevalence_long["rater"]
    .str.replace("prevalence_", "", regex=False)
)

plt.figure(figsize=(6.6, 3.7))
axis = plt.gca()

for rater in EXPECTED_RATERS:
    subset = prevalence_long.loc[
        prevalence_long["rater"].eq(rater)
    ]
    axis.plot(
        subset["signal_family"],
        100 * subset["prevalence"],
        marker="o",
        label=rater,
    )

axis.set_ylabel("Positive ratings (%)")
axis.set_xlabel("")
axis.set_title(
    "Human family-label prevalence by rater",
    loc="left",
    fontweight="bold",
)
axis.legend(frameon=False, ncol=4)
axis.grid(axis="y", alpha=0.25)

save_current_figure(
    "Supplementary_human_family_prevalence_by_rater"
)
plt.show()


# --------------------------------------------------------------
# Figure 2: AC1 and Fleiss kappa
# --------------------------------------------------------------
plot_data = family_reliability.loc[
    family_reliability["signal_family"].isin(
        ACQUISITION_FAMILIES
    )
].copy()

positions = np.arange(len(plot_data))

plt.figure(figsize=(6.2, 3.8))
axis = plt.gca()

axis.errorbar(
    plot_data["gwet_ac1"],
    positions - 0.10,
    xerr=np.vstack([
        plot_data["gwet_ac1"]
        - plot_data["gwet_ac1_ci_low"],
        plot_data["gwet_ac1_ci_high"]
        - plot_data["gwet_ac1"],
    ]),
    fmt="o",
    capsize=2,
    label="Gwet AC1",
)

axis.errorbar(
    plot_data["fleiss_kappa"],
    positions + 0.10,
    xerr=np.vstack([
        plot_data["fleiss_kappa"]
        - plot_data["fleiss_kappa_ci_low"],
        plot_data["fleiss_kappa_ci_high"]
        - plot_data["fleiss_kappa"],
    ]),
    fmt="s",
    capsize=2,
    label="Fleiss kappa",
)

axis.axvline(0, linewidth=0.8)
axis.set_yticks(positions)
axis.set_yticklabels(plot_data["signal_family"])
axis.invert_yaxis()
axis.set_xlabel("Agreement coefficient (95% participant-bootstrap CI)")
axis.set_title(
    "Human family-label reliability",
    loc="left",
    fontweight="bold",
)
axis.legend(frameon=False)

save_current_figure(
    "Supplementary_human_family_reliability_coefficients"
)
plt.show()


# --------------------------------------------------------------
# Figure 3: positive versus negative agreement
# --------------------------------------------------------------
plt.figure(figsize=(5.2, 4.4))
axis = plt.gca()

axis.scatter(
    plot_data[
        "mean_pairwise_positive_agreement"
    ],
    plot_data[
        "mean_pairwise_negative_agreement"
    ],
)

for row in plot_data.itertuples(index=False):
    axis.annotate(
        row.signal_family,
        (
            row.mean_pairwise_positive_agreement,
            row.mean_pairwise_negative_agreement,
        ),
        xytext=(4, 4),
        textcoords="offset points",
    )

axis.set_xlim(-0.02, 1.02)
axis.set_ylim(-0.02, 1.02)
axis.set_xlabel("Mean pairwise positive agreement")
axis.set_ylabel("Mean pairwise negative agreement")
axis.set_title(
    "Positive and negative agreement are not interchangeable",
    loc="left",
    fontweight="bold",
)
axis.grid(alpha=0.25)

save_current_figure(
    "Supplementary_human_family_positive_negative_agreement"
)
plt.show()


# --------------------------------------------------------------
# Figure 4: temporal Dice at 0 ms
# --------------------------------------------------------------
temporal_plot = family_temporal.loc[
    family_temporal["tolerance_label"].eq(
        "primary_0ms"
    )
    & family_temporal["outcome"].isin(
        ACQUISITION_FAMILIES
    )
].copy()

plt.figure(figsize=(6.2, 3.6))
axis = plt.gca()

axis.bar(
    temporal_plot["outcome"],
    temporal_plot["pooled_temporal_dice"],
)

axis.set_ylim(0, 1)
axis.set_ylabel("Pooled temporal Dice")
axis.set_xlabel("")
axis.set_title(
    "Continuous-time overlap among human raters",
    loc="left",
    fontweight="bold",
)
axis.grid(axis="y", alpha=0.25)

save_current_figure(
    "Supplementary_human_family_temporal_overlap"
)
plt.show()


# --------------------------------------------------------------
# Figure 5: item reliability map
# --------------------------------------------------------------
item_plot = item_reliability[
    [
        "item",
        "gwet_ac1",
        "mean_pairwise_positive_agreement",
        "mean_pairwise_negative_agreement",
    ]
].copy()

item_plot = item_plot.sort_values(
    [
        "mean_pairwise_positive_agreement",
        "gwet_ac1",
    ],
    ascending=False,
)

matrix = item_plot[
    [
        "gwet_ac1",
        "mean_pairwise_positive_agreement",
        "mean_pairwise_negative_agreement",
    ]
].to_numpy()

plt.figure(
    figsize=(
        6.8,
        max(5.8, 0.23 * len(item_plot)),
    )
)
axis = plt.gca()

image = axis.imshow(
    matrix,
    aspect="auto",
    vmin=0,
    vmax=1,
)

axis.set_xticks(np.arange(3))
axis.set_xticklabels(
    [
        "Gwet AC1",
        "Positive agreement",
        "Negative agreement",
    ],
    rotation=20,
    ha="right",
)

axis.set_yticks(
    np.arange(len(item_plot))
)
axis.set_yticklabels(
    item_plot["item"]
)
axis.set_title(
    "Human item-level reliability",
    loc="left",
    fontweight="bold",
)

colorbar = plt.colorbar(
    image,
    ax=axis,
)
colorbar.set_label("Agreement")

save_current_figure(
    "Supplementary_human_item_reliability"
)
plt.show()


## 15. Completion checks and run manifest

In [ ]:
overall_family_row = overall_reliability.loc[
    overall_reliability["analysis_level"].eq(
        "Six acquisition families"
    )
].iloc[0]

overall_item_row = overall_reliability.loc[
    overall_reliability["analysis_level"].eq(
        "Specific annotation items"
    )
].iloc[0]

blocking_checks = pd.DataFrame([
    {
        "check": "Expected four raters represented",
        "passed": set(folder_summary["rater"]) == set(EXPECTED_RATERS),
        "detail": " | ".join(
            sorted(folder_summary["rater"])
        ),
    },
    {
        "check": "Complete four-rater subset contains 69 recordings",
        "passed": len(reliability_inventory) == 69,
        "detail": f"recordings={len(reliability_inventory)}",
    },
    {
        "check": "All family presence matrices complete",
        "passed": all(
            matrix.shape == (69, 4)
            for matrix, _ in family_matrices.values()
        ),
        "detail": f"families={len(family_matrices)}",
    },
    {
        "check": "All item presence matrices complete",
        "passed": all(
            matrix.shape == (69, 4)
            for matrix, _ in item_matrices.values()
        ),
        "detail": f"items={len(item_matrices)}",
    },
    {
        "check": "Primary human interval reconstruction uses 0 ms",
        "passed": PRIMARY_HUMAN_MERGE_TOLERANCE_SEC == 0.0,
        "detail": (
            f"primary={PRIMARY_HUMAN_MERGE_TOLERANCE_SEC}; "
            f"sensitivity={SENSITIVITY_HUMAN_MERGE_TOLERANCE_SEC}"
        ),
    },
    {
        "check": "Overall family agreement reproduces prior audit",
        "passed": np.isclose(
            overall_family_row["observed_agreement"],
            0.781,
            atol=0.01,
        ),
        "detail": (
            f"observed={overall_family_row['observed_agreement']:.4f}"
        ),
    },
    {
        "check": "Overall family Gwet AC1 reproduces prior audit",
        "passed": np.isclose(
            overall_family_row["gwet_ac1"],
            0.643,
            atol=0.01,
        ),
        "detail": (
            f"observed={overall_family_row['gwet_ac1']:.4f}"
        ),
    },
    {
        "check": "Overall family Fleiss kappa reproduces prior audit",
        "passed": np.isclose(
            overall_family_row["fleiss_kappa"],
            0.432,
            atol=0.02,
        ),
        "detail": (
            f"observed={overall_family_row['fleiss_kappa']:.4f}"
        ),
    },
    {
        "check": "Overall item agreement is approximately 90 percent",
        "passed": np.isclose(
            overall_item_row["observed_agreement"],
            0.90,
            atol=0.02,
        ),
        "detail": (
            f"observed={overall_item_row['observed_agreement']:.4f}"
        ),
    },
    {
        "check": "No machine comparison performed",
        "passed": not ALLOW_MACHINE_COMPARISON,
        "detail": "human-reference analysis only",
    },
])

display(blocking_checks)

blocking_checks.to_csv(
    LOG_DIR / "stage1b_blocking_completion_checklist.csv",
    index=False,
)

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "analysis_root": str(ANALYSIS_ROOT),
    "stage1_root": str(STAGE1_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "expected_raters": EXPECTED_RATERS,
    "complete_four_rater_recordings": len(reliability_inventory),
    "approximate_reliability_participants": int(
        reliability_inventory[
            "reliability_subject_id"
        ].nunique()
    ),
    "acquisition_families": ACQUISITION_FAMILIES,
    "annotation_items": len(item_matrices),
    "primary_human_merge_tolerance_sec": (
        PRIMARY_HUMAN_MERGE_TOLERANCE_SEC
    ),
    "sensitivity_human_merge_tolerance_sec": (
        SENSITIVITY_HUMAN_MERGE_TOLERANCE_SEC
    ),
    "bootstrap_replicates": N_BOOTSTRAP,
    "final_run": FINAL_RUN,
    "overall_family_observed_agreement": float(
        overall_family_row["observed_agreement"]
    ),
    "overall_family_fleiss_kappa": float(
        overall_family_row["fleiss_kappa"]
    ),
    "overall_family_gwet_ac1": float(
        overall_family_row["gwet_ac1"]
    ),
    "overall_item_observed_agreement": float(
        overall_item_row["observed_agreement"]
    ),
    "machine_comparison_enabled": ALLOW_MACHINE_COMPARISON,
    "python": sys.version,
    "platform": platform.platform(),
}

(
    MANIFEST_DIR
    / "stage1b_run_manifest.json"
).write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

if not blocking_checks["passed"].all():
    print(
        "\nSTAGE 1B COMPLETED WITH BLOCKING ITEMS. "
        "Do not proceed to machine correspondence."
    )
else:
    print(
        "\nSTAGE 1B COMPLETED. "
        "The human reference is ready for scientific review. "
        "Machine correspondence remains disabled."
    )


---

## Stage-1B stop

Do not fit signal–human correspondence models yet.

Review:

1. family-level positive and negative agreement;
2. item-level agreement and prevalence;
3. Cochran-Q evidence of rater-specific thresholds;
4. each rater’s agreement with the majority of the other three;
5. temporal Dice and Jaccard under the 0 ms reconstruction;
6. the change under the 50 ms reconstruction sensitivity;
7. provisional human-reference roles.

Stage 2 will then freeze:

- which human family or item outcomes are confirmatory, exploratory, or descriptive;
- the exact rater-adjusted model specification;
- leave-one-rater-out requirements;
- which signal indicators are construct-matched to each retained human outcome;
- whether any event-level QDIST or QTEMP analysis is supportable.
